# M42 lab solution: the global-question test

**Brief.** Global-question test: ask ten "summarise / compare across the whole corpus" questions and compare flat RAG, RAPTOR and LightRAG.

Three systems over the same Larkspur handbook chunks, with the same chat model and embeddings:

1. **Flat RAG.** Top 4 chunks by embedding similarity, then one answer call.
2. **RAPTOR** (notebook `01a_raptor.ipynb`). Chunks are embedded, clustered with a Gaussian mixture and summarised, level by level, into a tree; retrieval searches the collapsed tree (chunks and summaries together).
3. **LightRAG** (notebook `03a_light_rag.ipynb`). One extraction call per chunk builds an entity and relationship graph; a question is split into low-level and high-level keywords, which search an entity index and a relationship index, and the answer is written from the matched entities, relationships and their source chunks.

An LLM judge scores each answer from 1 to 5 against the reference answer in `../data/global_questions.jsonl`. The judge prompt is printed below, so the grading can be checked.

## Setup

In [ ]:
import sys
import time
from collections import defaultdict
from pathlib import Path

import networkx as nx
import numpy as np
from dotenv import load_dotenv
from langchain_core.callbacks import UsageMetadataCallbackHandler
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field
from sklearn.mixture import GaussianMixture

load_dotenv()
sys.path.insert(0, str(Path("../../../../src").resolve()))
from helpers.utils import get_embeddings, get_experientiallabs_llm  # noqa: E402
from lab_kit import load_handbook, load_jsonl  # noqa: E402

DATA = Path("../data")
llm = get_experientiallabs_llm()
embeddings = get_embeddings()
MODEL = llm.model_name

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [Document(page_content=c, metadata={"source": p["source"], "chunk_id": i})
          for i, (p, c) in enumerate((p, c) for p in load_handbook(DATA / "larkspur" / "handbook")
                                     for c in splitter.split_text(p["text"]))]
questions = load_jsonl(DATA / "global_questions.jsonl")
print("model:", MODEL, "|", len(chunks), "chunks |", len(questions), "global questions")


def answer_from(context: str, question: str) -> str:
    return llm.invoke([
        SystemMessage("Answer the question from the context only. The question may ask you to summarise or compare "
                      "across the whole handbook: cover every relevant point the context gives.\n\nContext:\n" + context),
        HumanMessage(question),
    ]).content


build = {}  # indexing cost per system: seconds and tokens

## System 1: flat RAG

In [ ]:
start = time.perf_counter()
flat_store = InMemoryVectorStore.from_documents(chunks, embeddings)
build["flat RAG"] = {"seconds": time.perf_counter() - start, "llm_tokens": 0}


def flat_rag(question: str, k: int = 4) -> str:
    docs = flat_store.similarity_search(question, k=k)
    return answer_from("\n---\n".join(d.page_content for d in docs), question)

## System 2: RAPTOR

Level 0 is the chunks. Each level clusters the level below with a Gaussian mixture (one cluster per five nodes, at least two) and writes one summary per cluster. The tree stops at three levels or when a level has three nodes or fewer. Retrieval searches all nodes at once (the collapsed tree), so a global question can match a summary and a detail question can match a chunk.

In [ ]:
MAX_LEVELS = 3


def summarise(texts: list[str], cb) -> str:
    return llm.invoke([
        SystemMessage("Summarise these passages from a cloud host's customer handbook. Keep every number, limit and "
                      "plan name."),
        HumanMessage("\n---\n".join(texts)),
    ], config={"callbacks": [cb]}).content


def build_raptor(base: list[Document]) -> tuple[list[Document], UsageMetadataCallbackHandler]:
    cb = UsageMetadataCallbackHandler()
    nodes, level = list(base), list(base)
    for depth in range(1, MAX_LEVELS + 1):
        if len(level) <= 3:
            break
        vectors = np.array(embeddings.embed_documents([d.page_content for d in level]))
        k = max(2, len(level) // 5)
        labels = GaussianMixture(n_components=k, random_state=0).fit_predict(vectors)
        level = [
            Document(page_content=summarise([d.page_content for d, lab in zip(level, labels) if lab == c], cb),
                     metadata={"level": depth, "cluster": int(c)})
            for c in sorted(set(labels))
        ]
        nodes += level
        print(f"level {depth}: {len(level)} summaries")
    return nodes, cb


start = time.perf_counter()
raptor_nodes, raptor_cb = build_raptor(chunks)
raptor_store = InMemoryVectorStore.from_documents(raptor_nodes, embeddings)
build["RAPTOR"] = {"seconds": time.perf_counter() - start,
                   "llm_tokens": sum(u.get("total_tokens", 0) for u in raptor_cb.usage_metadata.values())}


def raptor(question: str, k: int = 6) -> str:
    docs = raptor_store.similarity_search(question, k=k)
    return answer_from("\n---\n".join(d.page_content for d in docs), question)

## System 3: LightRAG

The extraction schema and the answer prompt follow the module's `light_rag.py`; the graph is a `networkx` graph and the two indexes are in-memory vector stores.

In [ ]:
class Entity(BaseModel):
    name: str = Field(description="Canonical name of the entity, capitalized so it can be merged across chunks")
    type: str = Field(description="Entity type, e.g. plan, feature, policy, region, team, limit")
    description: str = Field(description="Concise description of the entity based ONLY on the text")


class Relationship(BaseModel):
    source: str = Field(description="Name of the source entity")
    target: str = Field(description="Name of the target entity")
    description: str = Field(description="How the two entities are related, based on the text")
    keywords: str = Field(description="High-level keywords summarizing the nature of the relationship")


class GraphExtraction(BaseModel):
    entities: list[Entity] = Field(default_factory=list)
    relationships: list[Relationship] = Field(default_factory=list)


class QueryKeywords(BaseModel):
    low_level: list[str] = Field(description="Specific entities, names, or concrete details in the question")
    high_level: list[str] = Field(description="Broad themes or overarching topics the question is about")


EXTRACT = ("Extract a knowledge graph from the text: the key entities (name, type, short description) and the "
           "relationships between them (source, target, description, keywords). Use only the text. Keep names "
           "consistent so the same entity merges across chunks.\n\nText:\n{text}")


def build_lightrag(docs: list[Document]):
    cb = UsageMetadataCallbackHandler()
    extractor = llm.with_structured_output(GraphExtraction)
    graph, entity_chunks = nx.Graph(), defaultdict(set)
    for d in docs:
        cid = d.metadata["chunk_id"]
        try:
            ex = extractor.invoke(EXTRACT.format(text=d.page_content), config={"callbacks": [cb]})
        except Exception as exc:  # noqa: BLE001
            print("skipped chunk", cid, exc)
            continue
        for e in ex.entities:
            key = e.name.strip().lower()
            if key:
                graph.add_node(key, name=e.name, type=e.type)
                graph.nodes[key].setdefault("descriptions", []).append(e.description)
                entity_chunks[key].add(cid)
        for r in ex.relationships:
            s, t = r.source.strip().lower(), r.target.strip().lower()
            if not s or not t or s == t:
                continue
            for key, raw in ((s, r.source), (t, r.target)):
                if key not in graph:
                    graph.add_node(key, name=raw, type="unknown", descriptions=[])
                entity_chunks[key].add(cid)
            if graph.has_edge(s, t):
                graph.edges[s, t]["descriptions"].append(r.description)
                graph.edges[s, t]["keywords"].append(r.keywords)
                graph.edges[s, t]["chunk_ids"].add(cid)
            else:
                graph.add_edge(s, t, descriptions=[r.description], keywords=[r.keywords], chunk_ids={cid})
    for _, n in graph.nodes(data=True):
        n["description"] = " ".join(dict.fromkeys(n.get("descriptions", [])))
    for _, _, e in graph.edges(data=True):
        e["description"] = " ".join(dict.fromkeys(e["descriptions"]))
        e["keywords"] = ", ".join(dict.fromkeys(e["keywords"]))
    entity_index = InMemoryVectorStore.from_documents(
        [Document(page_content=f"{n['name']} ({n['type']}): {n['description']}", metadata={"entity": k})
         for k, n in graph.nodes(data=True)], embeddings)
    relation_index = InMemoryVectorStore.from_documents(
        [Document(page_content=f"{graph.nodes[u]['name']} -- {graph.nodes[v]['name']} | keywords: {e['keywords']} | "
                               f"{e['description']}", metadata={"source": u, "target": v})
         for u, v, e in graph.edges(data=True)], embeddings)
    return graph, entity_chunks, entity_index, relation_index, cb


start = time.perf_counter()
kg, entity_chunks, entity_index, relation_index, lightrag_cb = build_lightrag(chunks)
build["LightRAG"] = {"seconds": time.perf_counter() - start,
                     "llm_tokens": sum(u.get("total_tokens", 0) for u in lightrag_cb.usage_metadata.values())}
print(kg.number_of_nodes(), "entities,", kg.number_of_edges(), "relationships")

In [ ]:
ANSWER_PROMPT = """Answer the question using ONLY the context below. The context comes from a knowledge graph
(entities and their relationships) plus the original source passages.

### Entities
{entities}

### Relationships
{relations}

### Source passages
{chunks}

Question: {question}

Answer:"""
keyword_llm = llm.with_structured_output(QueryKeywords)


def lightrag(question: str, k_entities: int = 4, k_relations: int = 4) -> str:
    kws = keyword_llm.invoke(f"Extract low-level and high-level keywords from this question: {question}")
    ents, edges, cids = set(), set(), set()
    for hit in entity_index.similarity_search(", ".join(kws.low_level) or question, k=k_entities):
        e = hit.metadata["entity"]
        ents.add(e)
        cids |= entity_chunks[e]
        edges |= {tuple(sorted((e, n))) for n in kg.neighbors(e)}
    for hit in relation_index.similarity_search(", ".join(kws.high_level) or question, k=k_relations):
        s, t = hit.metadata["source"], hit.metadata["target"]
        edges.add(tuple(sorted((s, t))))
        ents |= {s, t}
        cids |= kg.edges[s, t]["chunk_ids"]
    prompt = ANSWER_PROMPT.format(
        entities="\n".join(f"- {kg.nodes[e]['name']}: {kg.nodes[e]['description']}" for e in ents),
        relations="\n".join(f"- {kg.nodes[u]['name']} -- {kg.nodes[v]['name']}: {kg.edges[u, v]['description']}"
                            for u, v in edges if kg.has_edge(u, v)),
        chunks="\n---\n".join(chunks[c].page_content for c in sorted(cids)),
        question=question,
    )
    return llm.invoke(prompt).content

## The judge

The judge sees the question, the reference answer and one system's answer, never the system's name. It scores coverage of the reference's points and penalises claims the reference contradicts.

In [ ]:
JUDGE_PROMPT = """You grade answers to questions about a company's customer handbook.

Question: {question}

Reference answer (complete and correct):
{reference}

Answer to grade:
{answer}

Score the answer from 1 to 5:
5 = covers every point of the reference and contradicts none
4 = covers most points, contradicts none
3 = covers about half the points, or one minor error
2 = covers few points, or a clear error
1 = misses the question or is mostly wrong
Reply with the score and one sentence naming what is missing or wrong."""


class Verdict(BaseModel):
    score: int = Field(ge=1, le=5)
    reason: str


judge = llm.with_structured_output(Verdict)
print(JUDGE_PROMPT)

## Run the ten questions

In [ ]:
SYSTEMS = {"flat RAG": flat_rag, "RAPTOR": raptor, "LightRAG": lightrag}
rows = []
for q in questions:
    for name, system in SYSTEMS.items():
        start = time.perf_counter()
        answer = system(q["question"])
        seconds = time.perf_counter() - start
        v = judge.invoke(JUDGE_PROMPT.format(question=q["question"], reference=q["reference_answer"], answer=answer))
        rows.append({"id": q["id"], "system": name, "score": v.score, "reason": v.reason,
                     "query_s": round(seconds, 2), "answer": answer})
    print(q["id"], [(r["system"], r["score"]) for r in rows[-3:]])

## Results

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
print("model:", MODEL, "| embeddings:", type(embeddings).__name__, "| run:", time.strftime("%Y-%m-%d"))
print(df.pivot(index="id", columns="system", values="score").to_markdown())
summary = df.groupby("system").agg(mean_score=("score", "mean"), median_query_s=("query_s", "median"))
summary["index_s"] = [round(build[s]["seconds"], 1) for s in summary.index]
summary["index_llm_tokens"] = [build[s]["llm_tokens"] for s in summary.index]
print()
print(summary.round(2).to_markdown())

In [ ]:
# The judge's reasons for every answer below 4, to read alongside the scores.
for r in df[df["score"] < 4].itertuples():
    print(f"{r.id} {r.system} ({r.score}): {r.reason}")

## What this shows

- Flat RAG retrieves four chunks, and a global question needs facts from up to seven handbook files, so its score is capped by what fits in those four chunks.
- RAPTOR pays for its summaries once, at indexing time (`index_llm_tokens`), and a global question can then match a summary that already combines several sections.
- LightRAG pays one extraction call per chunk. Its high-level keywords search relationships rather than passages, which is the route a "compare across the handbook" question needs; check the per-question scores to see whether that route found the right sections.
- An LLM judge is a measuring instrument with its own errors. Read the reasons printed above before trusting a difference of less than one point.